# 🌲 Template: Modelos de Ensamble

Plantilla reutilizable para entrenar y comparar modelos de ensamble.  
Cubre **Bagging**, **Boosting** y **Stacking** — de conceptual a implementación práctica.

**Posición en el flujo de trabajo:** `06-modelado/`

---

## ¿Cómo usar esta plantilla?

1. Configurá las variables en la celda de configuración
2. Ejecutá el bloque del algoritmo que necesitás
3. Los ejemplos conceptuales usan datasets sintéticos — los bloques de "Tu proyecto" usan `df_clean`

**Cuándo usar cada familia:**

| Familia | Problema que resuelve | Algoritmos |
|---|---|---|
| Bagging | Alta varianza (overfitting) | Random Forest, BaggingClassifier/Regressor |
| Boosting | Alto sesgo (underfitting) | AdaBoost, Gradient Boosting, XGBoost, LightGBM |
| Stacking | Combinar fortalezas de modelos distintos | StackingClassifier/Regressor |

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, r2_score, mean_squared_error
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

from sklearn.ensemble import (
    BaggingClassifier, BaggingRegressor,
    RandomForestClassifier, RandomForestRegressor,
    AdaBoostClassifier, AdaBoostRegressor,
    GradientBoostingRegressor,
    StackingClassifier, StackingRegressor
)

# Datasets sintéticos para ejemplos conceptuales
from sklearn.datasets import (
    make_classification, make_regression,
    load_iris, load_wine, load_breast_cancer,
    load_diabetes, fetch_california_housing
)

In [ ]:
# ── Configuración — modificar para cada proyecto ───────────
TARGET       = 'variable_objetivo'  # ← CAMBIAR
TIPO         = 'clasificacion'      # ← CAMBIAR: 'clasificacion' o 'regresion'
TEST_SIZE    = 0.2
RANDOM_STATE = 42
# ───────────────────────────────────────────────────────────

# Descomenta cuando uses con df_clean real:
# X = df_clean.drop(columns=[TARGET])
# y = df_clean[TARGET]
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
# )
# print(f'Train: {X_train.shape} | Test: {X_test.shape}')

---
# BLOQUE 1 — Bagging (Bootstrap Aggregating)

Bagging utiliza bootstrap para crear múltiples subconjuntos de datos y entrena un modelo en cada subconjunto. Luego, combina los resultados para obtener una predicción final.

**Objetivo:** Reducir la varianza de los modelos individuales.  
**Ventaja:** Especialmente útil para modelos inestables como los árboles de decisión.

**Reducción de la varianza en modelos**

Bagging reduce la dependencia de un único modelo y suaviza las predicciones al promediar (para regresión) o votar (para clasificación) las salidas de múltiples modelos.

---

## ¿Qué es el bootstrap?

El bootstrap es un método estadístico basado en el remuestreo aleatorio **con reemplazo**. Se utiliza para estimar propiedades de una población (como la media, la varianza o intervalos de confianza) al generar múltiples muestras a partir de un conjunto de datos original.

- No asume distribuciones específicas de los datos
- Funciona bien con conjuntos de datos pequeños
- Ayuda a estimar la variabilidad y la incertidumbre de un modelo
- **Remuestreo con reemplazo:** cada observación puede aparecer más de una vez en una muestra generada

**Ejemplo conceptual:**

Dataset: `[2, 4, 6, 8, 10]`

Muestra 1: `[4, 10, 2, 4, 8]` → Media = 5.6  
Muestra 2: `[6, 6, 8, 10, 2]` → Media = 6.4

Se repite el proceso múltiples veces para estimar propiedades del conjunto original.

## 1.1 — BaggingClassifier (Ejemplo conceptual)

In [ ]:
# Dataset sintético para ejemplo conceptual
X_ej, y_ej = make_classification(n_samples=1000, n_features=10, random_state=RANDOM_STATE)
X_train_ej, X_test_ej, y_train_ej, y_test_ej = train_test_split(
    X_ej, y_ej, test_size=0.3, random_state=RANDOM_STATE
)

# Bagging con Árboles de Decisión
bagging_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=50,
    random_state=RANDOM_STATE
)
bagging_clf.fit(X_train_ej, y_train_ej)
y_pred_ej = bagging_clf.predict(X_test_ej)
print("Precisión BaggingClassifier (árbol):", accuracy_score(y_test_ej, y_pred_ej))

# Bagging con KNN
bagging_knn = BaggingClassifier(
    estimator=KNeighborsClassifier(n_neighbors=5),
    n_estimators=10,
    random_state=RANDOM_STATE
)
bagging_knn.fit(X_train_ej, y_train_ej)
y_pred_knn = bagging_knn.predict(X_test_ej)
print("Precisión BaggingClassifier (KNN):  ", accuracy_score(y_test_ej, y_pred_knn))

## 1.2 — BaggingClassifier (Tu proyecto)

In [ ]:
# ── Configuración ──────────────────────────────────────────
N_ESTIMATORS_BAG_CLF = 50          # ← CAMBIAR
ESTIMADOR_BASE       = DecisionTreeClassifier()  # ← CAMBIAR si querés otro estimador
# ───────────────────────────────────────────────────────────

modelo_bag_clf = BaggingClassifier(
    estimator=ESTIMADOR_BASE,
    n_estimators=N_ESTIMATORS_BAG_CLF,
    random_state=RANDOM_STATE
)
modelo_bag_clf.fit(X_train, y_train)
y_pred = modelo_bag_clf.predict(X_test)

print(f"Precisión BaggingClassifier: {accuracy_score(y_test, y_pred):.4f}")

## 1.3 — BaggingRegressor (Ejemplo conceptual)

El Bagging Regressor es la variante de Bagging para tareas de regresión. La predicción final se obtiene **promediando** las predicciones de cada estimador base.

In [ ]:
# Dataset sintético
X_ej, y_ej = make_regression(n_samples=1000, n_features=10, noise=0.1, random_state=RANDOM_STATE)
X_train_ej, X_test_ej, y_train_ej, y_test_ej = train_test_split(
    X_ej, y_ej, test_size=0.3, random_state=RANDOM_STATE
)

bagging_reg = BaggingRegressor(
    estimator=DecisionTreeRegressor(),
    n_estimators=25,
    random_state=RANDOM_STATE
)
bagging_reg.fit(X_train_ej, y_train_ej)
y_pred_ej = bagging_reg.predict(X_test_ej)
print("R² BaggingRegressor:", r2_score(y_test_ej, y_pred_ej))

## 1.4 — BaggingRegressor (Tu proyecto)

In [ ]:
# ── Configuración ──────────────────────────────────────────
N_ESTIMATORS_BAG_REG = 25          # ← CAMBIAR
# ───────────────────────────────────────────────────────────

modelo_bag_reg = BaggingRegressor(
    estimator=DecisionTreeRegressor(),
    n_estimators=N_ESTIMATORS_BAG_REG,
    random_state=RANDOM_STATE
)
modelo_bag_reg.fit(X_train, y_train)
y_pred = modelo_bag_reg.predict(X_test)

print(f"R² BaggingRegressor: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

---
# BLOQUE 2 — Random Forest

Random Forest es un algoritmo de aprendizaje supervisado que usa Bagging con árboles de decisión. Cada árbol se entrena con un subconjunto aleatorio de datos **y** un subconjunto aleatorio de features.

**Pasos fundamentales:**
1. **Bootstrap Sampling:** subconjunto aleatorio del dataset con reemplazo
2. **Selección aleatoria de features:** en cada división del árbol, solo considera un subconjunto de variables
3. **Predicción final:** votación mayoritaria (clasificación) o promedio (regresión)

**Ventajas:**
- Reducción de la varianza al combinar múltiples árboles
- Menor riesgo de sobreajuste que un árbol solo
- Permite calcular importancia de features

## 2.1 — RandomForestClassifier (Ejemplo conceptual)

In [ ]:
iris = load_iris()
X_ej, y_ej = iris.data, iris.target
X_train_ej, X_test_ej, y_train_ej, y_test_ej = train_test_split(
    X_ej, y_ej, test_size=0.3, random_state=RANDOM_STATE
)

rf_clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_clf.fit(X_train_ej, y_train_ej)
y_pred_ej = rf_clf.predict(X_test_ej)
print("Precisión RandomForestClassifier (Iris):", accuracy_score(y_test_ej, y_pred_ej))

## 2.2 — RandomForestRegressor (Ejemplo conceptual)

In [ ]:
california = fetch_california_housing()
X_ej, y_ej = california.data, california.target
X_train_ej, X_test_ej, y_train_ej, y_test_ej = train_test_split(
    X_ej, y_ej, test_size=0.2, random_state=RANDOM_STATE
)

rf_reg = RandomForestRegressor(n_estimators=100, max_depth=3, random_state=RANDOM_STATE)
rf_reg.fit(X_train_ej, y_train_ej)
y_pred_ej = rf_reg.predict(X_test_ej)
print("MSE RandomForestRegressor (California Housing):", mean_squared_error(y_test_ej, y_pred_ej))

## 2.3 — Random Forest (Tu proyecto)

In [ ]:
# ── Configuración ──────────────────────────────────────────
N_ESTIMATORS_RF = 100   # ← CAMBIAR: más árboles = más robusto pero más lento
MAX_DEPTH_RF    = None  # ← CAMBIAR: None = sin límite, int = limitar profundidad
# ───────────────────────────────────────────────────────────

if TIPO == 'clasificacion':
    modelo_rf = RandomForestClassifier(
        n_estimators=N_ESTIMATORS_RF,
        max_depth=MAX_DEPTH_RF,
        random_state=RANDOM_STATE
    )
else:
    modelo_rf = RandomForestRegressor(
        n_estimators=N_ESTIMATORS_RF,
        max_depth=MAX_DEPTH_RF,
        random_state=RANDOM_STATE
    )

modelo_rf.fit(X_train, y_train)
y_pred = modelo_rf.predict(X_test)

if TIPO == 'clasificacion':
    print(f"Precisión Random Forest: {accuracy_score(y_test, y_pred):.4f}")
else:
    print(f"R²:   {r2_score(y_test, y_pred):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

# Importancia de features
importancias = pd.Series(
    modelo_rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("\nTop 10 features más importantes:")
print(importancias.head(10))

---
# BLOQUE 3 — Boosting

Boosting crea modelos de manera **secuencial**, donde cada modelo intenta corregir los errores del anterior.

**Objetivo:** Reducir el sesgo y mejorar la precisión.  
**Proceso:**
1. Entrenamos un modelo base
2. Evaluamos sus errores y asignamos mayores pesos a las observaciones mal predichas
3. Entrenamos un nuevo modelo para corregir esos errores
4. Combinamos todos los modelos para hacer predicciones finales

---
## 3.1 — AdaBoost (Adaptive Boosting)

AdaBoost ajusta los pesos de los clasificadores de manera iterativa. Cada nuevo modelo se ajusta a los errores cometidos por el modelo anterior. Las instancias mal clasificadas reciben mayor peso para que el siguiente modelo las priorice.

In [ ]:
# Ejemplo conceptual — Clasificación con Iris
iris = load_iris()
X_ej, y_ej = iris.data, iris.target
X_train_ej, X_test_ej, y_train_ej, y_test_ej = train_test_split(
    X_ej, y_ej, test_size=0.3, random_state=RANDOM_STATE
)

adaboost_clf = AdaBoostClassifier(n_estimators=50, random_state=RANDOM_STATE)
adaboost_clf.fit(X_train_ej, y_train_ej)
y_pred_ej = adaboost_clf.predict(X_test_ej)
print("Precisión AdaBoost (Iris):", accuracy_score(y_test_ej, y_pred_ej))

### AdaBoost — Tu proyecto

In [ ]:
# ── Configuración ──────────────────────────────────────────
N_ESTIMATORS_ADA = 50  # ← CAMBIAR
# ───────────────────────────────────────────────────────────

if TIPO == 'clasificacion':
    modelo_ada = AdaBoostClassifier(n_estimators=N_ESTIMATORS_ADA, random_state=RANDOM_STATE)
else:
    modelo_ada = AdaBoostRegressor(n_estimators=N_ESTIMATORS_ADA, random_state=RANDOM_STATE)

modelo_ada.fit(X_train, y_train)
y_pred = modelo_ada.predict(X_test)

if TIPO == 'clasificacion':
    print(f"Precisión AdaBoost: {accuracy_score(y_test, y_pred):.4f}")
else:
    print(f"R²:   {r2_score(y_test, y_pred):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

---
## 3.2 — Gradient Boosting

**Antes de ver Gradient Boosting — ¿Cómo funciona el descenso de gradiente?**

El descenso de gradiente es un algoritmo de optimización que minimiza una función objetivo (como el error del modelo) ajustando iterativamente los parámetros.

**Pasos:**
1. **Inicialización:** valores iniciales para los parámetros
2. **Cálculo del gradiente:** dirección y magnitud del cambio necesario
3. **Actualización:** $θ := θ - η∇J(θ)$ donde $η$ es la tasa de aprendizaje
4. **Repetición:** hasta que el error deja de mejorar

**Tipos de descenso de gradiente:**
- **Batch (BGD):** usa todo el dataset — preciso pero lento
- **Stochastic (SGD):** un ejemplo por iteración — rápido pero ruidoso
- **Mini-Batch:** equilibrio entre BGD y SGD

**En Gradient Boosting:** cada nuevo árbol se ajusta a los **residuos** del modelo anterior usando gradiente descendente para minimizar el error.

In [ ]:
# Ejemplo conceptual — Regresión con Diabetes
diabetes = load_diabetes()
X_ej, y_ej = diabetes.data, diabetes.target
X_train_ej, X_test_ej, y_train_ej, y_test_ej = train_test_split(
    X_ej, y_ej, test_size=0.3, random_state=RANDOM_STATE
)

gb_reg = GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE)
gb_reg.fit(X_train_ej, y_train_ej)
y_pred_ej = gb_reg.predict(X_test_ej)
print("R² Gradient Boosting (Diabetes):", r2_score(y_test_ej, y_pred_ej))

### Gradient Boosting — Tu proyecto

In [ ]:
# ── Configuración ──────────────────────────────────────────
N_ESTIMATORS_GB  = 100   # ← CAMBIAR
LEARNING_RATE_GB = 0.1   # ← CAMBIAR: valores menores requieren más árboles
MAX_DEPTH_GB     = 3     # ← CAMBIAR: árboles shallow son estándar en boosting
# ───────────────────────────────────────────────────────────

modelo_gb = GradientBoostingRegressor(
    n_estimators=N_ESTIMATORS_GB,
    learning_rate=LEARNING_RATE_GB,
    max_depth=MAX_DEPTH_GB,
    random_state=RANDOM_STATE
)
modelo_gb.fit(X_train, y_train)
y_pred = modelo_gb.predict(X_test)

print(f"R²:   {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

---
## 3.3 — XGBoost (Extreme Gradient Boosting)

XGBoost es una implementación optimizada de Gradient Boosting con características avanzadas:

- **Regularización L1 (Lasso) y L2 (Ridge):** controla la complejidad del modelo y evita sobreajuste
- **Manejo de valores faltantes:** aprende automáticamente qué hacer con NaN durante el entrenamiento
- **Paralelización:** entrenamiento más rápido en grandes datasets
- **Poda de árboles:** eliminación de ramas que no aportan valor

In [ ]:
import xgboost as xgb

# Ejemplo conceptual — Clasificación con Breast Cancer
cancer = load_breast_cancer()
X_ej, y_ej = cancer.data, cancer.target
X_train_ej, X_test_ej, y_train_ej, y_test_ej = train_test_split(
    X_ej, y_ej, test_size=0.3, random_state=RANDOM_STATE
)

xgb_clf = xgb.XGBClassifier(n_estimators=100, random_state=RANDOM_STATE, verbosity=0)
xgb_clf.fit(X_train_ej, y_train_ej)
y_pred_ej = xgb_clf.predict(X_test_ej)
print("Precisión XGBoost (Breast Cancer):", accuracy_score(y_test_ej, y_pred_ej))

### XGBoost — Tu proyecto

In [ ]:
# ── Configuración ──────────────────────────────────────────
N_ESTIMATORS_XGB  = 100   # ← CAMBIAR
LEARNING_RATE_XGB = 0.1   # ← CAMBIAR
MAX_DEPTH_XGB     = 6     # ← CAMBIAR: default de XGBoost es 6
# ───────────────────────────────────────────────────────────

if TIPO == 'clasificacion':
    modelo_xgb = xgb.XGBClassifier(
        n_estimators=N_ESTIMATORS_XGB,
        learning_rate=LEARNING_RATE_XGB,
        max_depth=MAX_DEPTH_XGB,
        random_state=RANDOM_STATE,
        verbosity=0
    )
else:
    modelo_xgb = xgb.XGBRegressor(
        n_estimators=N_ESTIMATORS_XGB,
        learning_rate=LEARNING_RATE_XGB,
        max_depth=MAX_DEPTH_XGB,
        random_state=RANDOM_STATE,
        verbosity=0
    )

modelo_xgb.fit(X_train, y_train)
y_pred = modelo_xgb.predict(X_test)

if TIPO == 'clasificacion':
    print(f"Precisión XGBoost: {accuracy_score(y_test, y_pred):.4f}")
else:
    print(f"R²:   {r2_score(y_test, y_pred):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

---
## 3.4 — LightGBM (Light Gradient Boosting Machine)

LightGBM está diseñado para ser más rápido y eficiente que otras implementaciones de boosting, ideal para grandes volúmenes de datos.

**Características avanzadas:**
- **Histogram-based learning:** usa histogramas para dividir los datos, acelerando el entrenamiento
- **Leaf-wise tree growth:** crece el árbol por la hoja con mayor reducción de error (vs. XGBoost que crece nivel por nivel)
- **Manejo de valores faltantes** nativo
- **Soporte para clasificación y regresión**

In [ ]:
import lightgbm as lgb

# Ejemplo conceptual — Clasificación con Iris
iris = load_iris()
X_ej = pd.DataFrame(iris.data, columns=iris.feature_names)
y_ej = iris.target
X_train_ej, X_test_ej, y_train_ej, y_test_ej = train_test_split(
    X_ej, y_ej, test_size=0.3, random_state=RANDOM_STATE
)

lgbm_clf = lgb.LGBMClassifier(n_estimators=100, random_state=RANDOM_STATE, verbose=-1)
lgbm_clf.fit(X_train_ej, y_train_ej)
y_pred_ej = lgbm_clf.predict(X_test_ej)
print("Precisión LightGBM (Iris):", accuracy_score(y_test_ej, y_pred_ej))

### LightGBM — Tu proyecto

In [ ]:
# ── Configuración ──────────────────────────────────────────
N_ESTIMATORS_LGBM  = 100   # ← CAMBIAR
LEARNING_RATE_LGBM = 0.1   # ← CAMBIAR
NUM_LEAVES_LGBM    = 31    # ← CAMBIAR: parámetro clave de LightGBM (default: 31)
# ───────────────────────────────────────────────────────────

if TIPO == 'clasificacion':
    modelo_lgbm = lgb.LGBMClassifier(
        n_estimators=N_ESTIMATORS_LGBM,
        learning_rate=LEARNING_RATE_LGBM,
        num_leaves=NUM_LEAVES_LGBM,
        random_state=RANDOM_STATE,
        verbose=-1
    )
else:
    modelo_lgbm = lgb.LGBMRegressor(
        n_estimators=N_ESTIMATORS_LGBM,
        learning_rate=LEARNING_RATE_LGBM,
        num_leaves=NUM_LEAVES_LGBM,
        random_state=RANDOM_STATE,
        verbose=-1
    )

modelo_lgbm.fit(X_train, y_train)
y_pred = modelo_lgbm.predict(X_test)

if TIPO == 'clasificacion':
    print(f"Precisión LightGBM: {accuracy_score(y_test, y_pred):.4f}")
else:
    print(f"R²:   {r2_score(y_test, y_pred):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

---
# BLOQUE 4 — Stacking y Blending

**Stacking:** combina múltiples modelos entrenando un **modelo meta** (de nivel superior) que usa las predicciones de los modelos base como features.

**Ventaja:** aprovecha la diversidad de modelos para mejorar la generalización.

**Blending:** similar a Stacking, pero usa una partición del dataset para entrenar el modelo meta (menor complejidad computacional).  
> Scikit-learn no tiene soporte directo para Blending — se implementa manualmente.

**Ejemplo conceptual:** combinar un árbol de decisión + KNN, y usar Regresión Logística como modelo meta.

## 4.1 — StackingClassifier (Ejemplo conceptual)

In [ ]:
X_ej, y_ej = make_classification(n_samples=1000, n_features=10, random_state=RANDOM_STATE)
X_train_ej, X_test_ej, y_train_ej, y_test_ej = train_test_split(
    X_ej, y_ej, test_size=0.3, random_state=RANDOM_STATE
)

# Modelos base
modelos_base = [
    ('arbol', DecisionTreeClassifier()),
    ('knn',   KNeighborsClassifier())
]

# Modelo meta
stacking_clf = StackingClassifier(
    estimators=modelos_base,
    final_estimator=LogisticRegression()
)
stacking_clf.fit(X_train_ej, y_train_ej)
y_pred_ej = stacking_clf.predict(X_test_ej)
print("Precisión StackingClassifier:", accuracy_score(y_test_ej, y_pred_ej))

## 4.2 — StackingRegressor (Ejemplo conceptual)

In [ ]:
X_ej, y_ej = make_regression(n_samples=1000, n_features=20, noise=0.1, random_state=RANDOM_STATE)
X_train_ej, X_test_ej, y_train_ej, y_test_ej = train_test_split(
    X_ej, y_ej, test_size=0.3, random_state=RANDOM_STATE
)

modelos_base = [
    ('regresion_lineal', LinearRegression()),
    ('knn',             KNeighborsRegressor())
]

stacking_reg = StackingRegressor(
    estimators=modelos_base,
    final_estimator=DecisionTreeRegressor(random_state=RANDOM_STATE)
)
stacking_reg.fit(X_train_ej, y_train_ej)
y_pred_ej = stacking_reg.predict(X_test_ej)
print("MSE StackingRegressor:", mean_squared_error(y_test_ej, y_pred_ej))

## 4.3 — Stacking (Tu proyecto)

In [ ]:
# ── Configuración ──────────────────────────────────────────
# Definí los modelos base y el meta-modelo según tu problema
# ───────────────────────────────────────────────────────────

if TIPO == 'clasificacion':
    modelos_base = [                             # ← CAMBIAR: elegí tus modelos base
        ('arbol', DecisionTreeClassifier(random_state=RANDOM_STATE)),
        ('knn',   KNeighborsClassifier())
    ]
    meta_modelo = LogisticRegression()           # ← CAMBIAR: elegí el meta-modelo
    modelo_stacking = StackingClassifier(
        estimators=modelos_base,
        final_estimator=meta_modelo
    )
else:
    modelos_base = [                             # ← CAMBIAR
        ('lineal', LinearRegression()),
        ('knn',    KNeighborsRegressor())
    ]
    meta_modelo = DecisionTreeRegressor(random_state=RANDOM_STATE)  # ← CAMBIAR
    modelo_stacking = StackingRegressor(
        estimators=modelos_base,
        final_estimator=meta_modelo
    )

modelo_stacking.fit(X_train, y_train)
y_pred = modelo_stacking.predict(X_test)

if TIPO == 'clasificacion':
    print(f"Precisión Stacking: {accuracy_score(y_test, y_pred):.4f}")
else:
    print(f"R²:   {r2_score(y_test, y_pred):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

---
# BLOQUE 5 — Comparación de Todos los Modelos

Tabla resumen conceptual:

| Característica | Bagging | Stacking | Blending | Boosting |
|---|---|---|---|---|
| **Objetivo** | Reducir varianza | Combinar con meta-modelo | Combinar con partición | Reducir sesgo |
| **Estrategia** | Modelos en paralelo | Meta-modelo aprende a combinar | Meta-modelo aprende a combinar | Secuencial, corrige errores |
| **Datos usados** | Muestras Bootstrap | Dataset completo para modelos base | Divide train para validación | Todo el conjunto, ajusta pesos |
| **Combinación** | Votación/Promedio | Meta-modelo | Meta-modelo | Pesos dinámicos |
| **Complejidad** | Menor | Mayor | Media | Alta |

In [ ]:
# Comparación numérica de todos los modelos entrenados en tu proyecto
# Ejecutar solo después de haber entrenado todos los modelos anteriores

resultados = {}

modelos = {
    'BaggingClassifier':  modelo_bag_clf  if TIPO == 'clasificacion' else modelo_bag_reg,
    'Random Forest':      modelo_rf,
    'AdaBoost':           modelo_ada,
    'Gradient Boosting':  modelo_gb,
    'XGBoost':            modelo_xgb,
    'LightGBM':           modelo_lgbm,
    'Stacking':           modelo_stacking
}

for nombre, modelo in modelos.items():
    y_pred = modelo.predict(X_test)
    if TIPO == 'clasificacion':
        resultados[nombre] = {'Accuracy': accuracy_score(y_test, y_pred)}
    else:
        resultados[nombre] = {
            'R²':   round(r2_score(y_test, y_pred), 4),
            'RMSE': round(np.sqrt(mean_squared_error(y_test, y_pred)), 4)
        }

df_resultados = pd.DataFrame(resultados).T
print("\n📊 Comparación de modelos:")
display(df_resultados.sort_values(
    'Accuracy' if TIPO == 'clasificacion' else 'R²',
    ascending=False
))

---
# 📋 Referencia Rápida

| Algoritmo | Familia | Tipo | Mejor para |
|---|---|---|---|
| BaggingClassifier/Regressor | Bagging | Ambos | Modelo base inestable con alta varianza |
| Random Forest | Bagging | Ambos | Primer modelo a probar, muy robusto |
| AdaBoost | Boosting | Ambos | Datasets balanceados, ruido bajo |
| Gradient Boosting | Boosting | Regresión | Control fino de underfitting |
| XGBoost | Boosting | Ambos | Competencias y proyectos con datos faltantes |
| LightGBM | Boosting | Ambos | Grandes datasets, velocidad de entrenamiento |
| Stacking | Ensamble avanzado | Ambos | Maximizar precisión combinando modelos distintos |

---
*Template de Modelos de Ensamble — ds-toolkit by andressonsino*  
*Posición en el flujo: `06-modelado/`*